In [ ]:
from flask import Flask, render_template, request, jsonify, session, redirect, url_for, flash
import pyttsx3
import schedule
import time
from threading import Thread
from datetime import datetime, timedelta
import csv
import os

app = Flask(__name__, template_folder="template", static_folder="static")
app.secret_key = "supersecretkey"  # Needed for sessions and flashing messages

engine = pyttsx3.init()

def speak(text, delay=1.5):
    print(f"Assistant: {text}")
    engine.say(text)
    engine.runAndWait()
    time.sleep(delay)

# Symptom database
symptoms_db = {
    "fever": "You may have a viral infection. Stay hydrated and rest.",
    "cough": "Try warm fluids and honey. If it persists, consult a doctor.",
    "cold": "Stay warm and drink hot fluids. Consider steam inhalation.",
    "headache": "You might be stressed or dehydrated. Drink water and relax.",
    "nausea": "Try ginger or peppermint tea. Rest is also important.",
    "fatigue": "Get adequate rest and hydrate well. Could be due to overexertion.",
    "sore throat": "Gargle with warm salt water and drink warm fluids.",
    "diarrhea": "Stay hydrated with ORS. Avoid oily and spicy food.",
    "vomiting": "Take small sips of water or ORS. Avoid solid food temporarily.",
    "stomach ache": "Lie down and rest. Avoid caffeine and spicy foods."
}

def check_symptom(user_input):
    best_match, score = process.extractOne(user_input.lower(), symptoms_list, scorer=fuzz.ratio)
    print(f"Fuzzy Match: '{best_match}' with score {score} for input '{user_input}'")

    matched_symptom_category = None
    for category, variations in symptoms_db.items():
        if isinstance(variations, str) and best_match == category.lower():
            matched_symptom_category = category
            break
        elif isinstance(variations, list) and best_match in [v.lower() for v in variations]:
            matched_symptom_category = category
            break

    if score >= 75 and matched_symptom_category in symptoms_db:
        remedy = symptoms_db[matched_symptom_category]  # Correctly retrieve the remedy
        log_interaction(user_input, remedy)
        return remedy
    else:
        unknown_msg = "I'm not sure about that. Could you describe your symptoms in more detail?"
        log_interaction(user_input, unknown_msg)
        return unknown_msg

# Auth routes
@app.route("/login", methods=["GET", "POST"])
def login():
    if request.method == "POST":
        username = request.form["username"]
        password = request.form["password"]

        if not os.path.exists("users.csv"):
            flash("No users registered yet.")
            return redirect(url_for("register"))

        with open("users.csv", newline='') as file:
            reader = csv.DictReader(file)
            for row in reader:
                if row["username"] == username and row["password"] == password:
                    session["username"] = username
                    return redirect(url_for("index"))
        flash("Invalid credentials!")
    return render_template("login.html")

@app.route("/register", methods=["GET", "POST"])
def register():
    if request.method == "POST":
        username = request.form["username"]
        password = request.form["password"]

        if os.path.exists("users.csv"):
            with open("users.csv", newline='') as file:
                reader = csv.DictReader(file)
                for row in reader:
                    if row["username"] == username:
                        flash("Username already exists!")
                        return redirect(url_for("register"))
        else:
            with open("users.csv", "w", newline='') as file:
                writer = csv.writer(file)
                writer.writerow(["username", "password"])

        with open("users.csv", "a", newline='') as file:
            writer = csv.writer(file)
            writer.writerow([username, password])
        flash("Registered successfully. Please log in.")
        return redirect(url_for("login"))

    return render_template("register.html")

@app.route("/logout")
def logout():
    session.pop("username", None)
    return redirect(url_for("login"))

# Routes
@app.route("/")
def index():
    if "username" not in session:
        return redirect(url_for("login"))
    return render_template("index.html")

@app.route("/speak_symptom", methods=["POST"])
def handle_speak_symptom():
    if "username" not in session:
        return jsonify({"response": "Please log in first."})
    data = request.get_json()
    symptom = data.get("symptom", "")
    response = check_symptom(symptom.lower())
    return jsonify({"response": response})

@app.route("/set_reminder", methods=["POST"])
def set_reminder():
    if "username" not in session:
        return jsonify({"status": "Please log in first."})
    data = request.get_json()
    medicine = data["medicine"]
    rtype = data["type"]
    status = "Reminder set."
    reminder_description = ""

    try:
        if rtype == "minutes":
            minutes = int(data["minutes"])
            reminder_time = datetime.now() + timedelta(minutes=minutes)
            time_str = reminder_time.strftime("%H:%M")
            schedule.every().day.at(time_str).do(speak, f"Take your medicine: {medicine}")
            reminder_description = f"in {minutes} minutes"
        elif rtype == "today":
            time_str = data["time"]
            schedule.every().day.at(time_str).do(speak, f"Take your medicine: {medicine}")
            reminder_description = f"today at {time_str}"
        elif rtype == "daily":
            time_str = data["time"]
            schedule.every().day.at(time_str).do(speak, f"Take your daily medicine: {medicine}")
            reminder_description = f"daily at {time_str}"
        else:
            status = "Invalid reminder type."
    except Exception as e:
        status = f"Error: {e}"

    if status == "Reminder set.":
        log_interaction("", "", f"{medicine} - {reminder_description}")

    return jsonify({"status": status})

def check_symptom(symptom):
    for key in symptoms_db:
        if key in symptom:
            remedy = symptoms_db[key]
            log_interaction(symptom, remedy)
            return remedy
    unknown_msg = "I couldn't find a match. Please consult a healthcare provider."
    log_interaction(symptom, unknown_msg)
    return unknown_msg

def log_interaction(symptoms, remedies, reminders="", feedback=""):
    username = session.get("username", "guest")
    with open("health_logs.csv", mode="a", newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow([
            username,
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            ", ".join(symptoms) if isinstance(symptoms, list) else symptoms,
            ", ".join(remedies) if isinstance(remedies, list) else remedies,
            ", ".join(reminders) if isinstance(reminders, list) else reminders,
            feedback
        ])

# @app.route("/view-logs")
# def view_logs():
#     if "username" not in session:
#         return redirect(url_for("login"))

#     logs = []
#     current_user = session["username"]
#     try:
#         with open("health_logs.csv", newline='', encoding='utf-8') as file:
#             reader = csv.reader(file)
#             for row in reader:
#                 if row and row[0] == current_user:
#                     logs.append({
#                         "timestamp": row[1],
#                         "symptoms": row[2],
#                         "remedies": row[3],
#                         "reminders": row[4],
#                         "feedback": row[5] if len(row) > 5 else ""
#                     })
#     except FileNotFoundError:
#         logs = []
#     return render_template("logs.html", logs=logs)

@app.route("/view-logs-json")
def view_logs_json():
    if "username" not in session:
        return jsonify({"error": "Please log in first."}), 401

    logs = []
    current_user = session["username"]
    try:
        with open("health_logs.csv", newline='', encoding='utf-8') as file:
            reader = csv.reader(file)
            for row in reader:
                if row and row[0] == current_user:
                    logs.append({
                        "timestamp": row[1],
                        "symptoms": row[2],
                        "remedies": row[3],
                        "reminders": row[4],
                        "feedback": row[5] if len(row) > 5 else ""
                    })
    except FileNotFoundError:
        pass  # Handle the case where the log file doesn't exist

    return jsonify(logs)

def run_scheduler():
    while True:
        schedule.run_pending()
        time.sleep(1)

Thread(target=run_scheduler, daemon=True).start()

import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt')

def extract_keywords(text):
    tokens = word_tokenize(text.lower())
    return tokens


# Route to render chatbot page
@app.route("/chatbot")
def chatbot():
    if "username" not in session:
        return redirect(url_for("login"))
    return render_template("chatbot.html")

# Route to handle AJAX chat messages
@app.route("/chat", methods=["POST"]) # <---- Changed the route to /chat
def chat():
    if "username" not in session:
        return jsonify({"response": "Please log in first."})

    data = request.get_json()
    user_input = data.get("message", "").strip()

    if not user_input:
        return jsonify({"response": "Please enter a message."}), 400

    keywords = extract_keywords(user_input)

    for word in keywords:
        if word in symptoms_db:
            remedy = symptoms_db[word]
            log_interaction(word, remedy)
            return jsonify({"response": remedy})

    unknown_response = "Sorry, I couldn't understand. Try rephrasing or be more specific."
    log_interaction(user_input, unknown_response)
    return jsonify({"response": unknown_response})
# if __name__ == "__main__":
#     app.run(debug=True)

from threading import Thread

def run_flask():
    app.run(debug=True, use_reloader=False)  # Disable reloader to avoid multiple launches

flask_thread = Thread(target=run_flask)
flask_thread.start()



In [ ]:
!pip install fuzzywuzzy